In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 04.1 · Transport parser + ablation V3

Este notebook documenta la iteración posterior a Day 04. Parte de las evidencias de `notebooks/11_day04_v3a_raw_contract_and_eval.ipynb` y responde dos preguntas:

1. Qué familia de señales de `V3_A` mete ruido o conserva señal útil cuando se aísla sobre `V2`.
2. Si la señal raw de `MENOR COSTE (CON TRANSPORTE)` puede convertirse en una variante pura reproducible sin romper el universo operativo.


In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT

registry_path = PROJECT_ROOT / "artifacts/public/metrics/final_baseline_vs_candidates.csv"
quality_day04_path = PROJECT_ROOT / "artifacts/public/data_quality_v3_context.json"
transport_report_path = PROJECT_ROOT / "artifacts/public/transport_parser_day041.json"
quality_day041_path = PROJECT_ROOT / "artifacts/public/data_quality_day041_ablation_matrix.json"
day041_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T095152Z_day041_ablation_run_summary.json"
metrics_dir = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306"

registry = pd.read_csv(registry_path)
quality_day04 = json.loads(quality_day04_path.read_text(encoding="utf-8"))
transport_report = json.loads(transport_report_path.read_text(encoding="utf-8"))
quality_day041 = json.loads(quality_day041_path.read_text(encoding="utf-8"))
day041_summary = json.loads(day041_summary_path.read_text(encoding="utf-8"))


## 1. Recap de Day 04

Day 04 confirmó que el bundle `V3_A` no superaba baseline y dejó una hipótesis abierta: quizá alguna familia aislada sí ayuda, pero mezcladas no. Además, el raw sí mostraba señal de transporte no aprovechada todavía.


In [3]:
display(Markdown("### Day 04 y Day 04.1 en el registro oficial"))
recap = registry.loc[registry["day_id"].isin(["Day 04", "Day 04.1"]) , [
    "day_id", "model_variant", "top1_hit", "top2_hit", "balanced_accuracy", "coverage", "promotion_decision"
]].copy()
display(recap)

transport_audit_day04 = pd.DataFrame([quality_day04["transport_audit"]])
display(Markdown("### Evidencia raw heredada de notebook 11"))
display(transport_audit_day04[["keyword_hits", "decision"]])


### Day 04 y Day 04.1 en el registro oficial

,day_id,model_variant,top1_hit,top2_hit,balanced_accuracy,coverage,promotion_decision
1,Day 04,V3_A_LR_smote_0.5_v1,0.739081,0.851880,0.853684,1.0,keep_baseline
2,Day 04,V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLL...,0.767186,0.855678,0.853684,1.0,keep_baseline
3,Day 04.1,V2_SOURCE_QUALITY_LR_smote_0.5_v1,0.762248,0.857197,0.855956,1.0,keep_baseline
4,Day 04.1,V2_DISPERSION_LR_smote_0.5_v1,0.764907,0.857197,0.865057,1.0,keep_baseline
5,Day 04.1,V2_COMPETITION_LR_smote_0.5_v1,0.764147,0.852260,0.867425,1.0,keep_baseline
6,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,0.773642,0.860235,0.865847,1.0,keep_baseline
7,Day 04.1,V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINIS...,0.789594,0.860235,0.865847,1.0,promote


### Evidencia raw heredada de notebook 11

,keyword_hits,decision
0,3413,candidate_parser_extension


## 2. Perfil de estabilidad del raw de transporte

El parser Day 04.1 se centra en `MENOR COSTE (CON TRANSPORTE)` desde `ofertas_raw_matrix_cells.csv` y audita sus layouts dominantes antes de intentar cualquier join al grano operativo.


In [4]:
display(Markdown("### Resumen del parser de transporte"))
transport_summary = pd.DataFrame([{
    "rows_output": transport_report["rows_output"],
    "files_parsed": transport_report["files_parsed"],
    "distinct_source_files_with_output": transport_report["distinct_source_files_with_output"],
    "xls_rows": transport_report["xls_rows"],
    "xlsx_rows": transport_report["xlsx_rows"],
    "rows_with_unknown_provider": transport_report["rows_with_unknown_provider"],
    "rows_with_multi_terminal_aggregate": transport_report["rows_with_multi_terminal_aggregate"],
}])
display(transport_summary)

display(Markdown("### Layouts dominantes"))
layout_counts = (
    pd.DataFrame(list(transport_report["layout_variant_counts"].items()), columns=["layout_variant", "rows"])
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)
display(layout_counts)

display(Markdown("### Estados del parser"))
status_counts = (
    pd.DataFrame(list(transport_report["status_counts"].items()), columns=["parser_status", "rows"])
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)
display(status_counts)

display(Markdown("### Muestra de filas parseadas"))
display(pd.DataFrame(transport_report["sample_rows"]).head(10))


### Resumen del parser de transporte

,rows_output,files_parsed,distinct_source_files_with_output,xls_rows,xlsx_rows,rows_with_unknown_provider,rows_with_multi_terminal_aggregate
0,98070,2496,2496,179,97891,2622,98035


### Layouts dominantes

,layout_variant,rows
0,tabla_anchor_row_27,95314
1,tabla_anchor_row_21,2537
2,tabla_anchor_row_25,219


### Estados del parser

,parser_status,rows
0,parsed_multi_terminal_aggregate,98035
1,parsed_unique_terminal,35


### Muestra de filas parseadas

,source_file,sheet_name,source_file_type,fecha_oferta,layout_variant,anchor_row_idx,provider_header_row_idx,product_header_row_idx,terminal_raw,terminal_canonico,producto_raw,producto_canonico,proveedor_raw,proveedor_candidato,transport_cost_value,parser_status,parser_run_id,parser_ts_utc
0,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_003,SUPPLIER_003,850.620,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
1,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_006,SUPPLIER_006,854.340,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
2,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_009,SUPPLIER_009,873.397,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
3,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_020,SUPPLIER_020,847.340,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
4,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_039,SUPPLIER_039,855.610,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
5,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_049,SUPPLIER_049,863.840,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
6,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_001,PRODUCT_001,SUPPLIER_054,SUPPLIER_054,849.730,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
7,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_002,PRODUCT_002,SUPPLIER_003,SUPPLIER_003,758.340,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
8,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_002,PRODUCT_002,SUPPLIER_006,SUPPLIER_006,764.340,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00
9,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMP...,Tabla,xls,2019-10-13,tabla_anchor_row_21,21,5,7,MULTI_TERMINAL_AGGREGATE,MULTI_TERMINAL_AGGREGATE,PRODUCT_002,PRODUCT_002,SUPPLIER_009,SUPPLIER_009,778.770,parsed_multi_terminal_aggregate,20260306T_day04_rerun_day041_parser,2026-03-06T12:24:05.071534+00:00


## 3. Contrato del parser y gate de transporte

El contrato operativo del parser queda restringido a señales deterministas y reproducibles. La regla de entrada a modelado no es “existe señal”, sino “existe señal con join estable, sin perder universo, y con cobertura mínima `>= 0.80` en train y test”.


In [5]:
display(Markdown("### Matriz de datasets Day 04.1"))
variant_rows = []
for variant_name, payload in quality_day041["variants"].items():
    coverage = payload.get("feature_coverage", {})
    min_train = min((item["coverage_train"] for item in coverage.values()), default=None)
    min_test = min((item["coverage_test"] for item in coverage.values()), default=None)
    variant_rows.append({
        "variant_name": variant_name,
        "status": payload.get("status"),
        "rows_output": payload.get("rows_output"),
        "events_output": payload.get("events_output"),
        "added_columns_n": len(payload.get("added_columns", [])),
        "min_coverage_train": min_train,
        "min_coverage_test": min_test,
        "events_with_invalid_positive_count": payload.get("events_with_invalid_positive_count"),
    })
variant_matrix = pd.DataFrame(variant_rows).sort_values(["status", "variant_name"]).reset_index(drop=True)
display(variant_matrix)

display(Markdown("### Detalle del gate de transport_only"))
transport_gate = quality_day041["variants"]["transport_only"]
display(pd.DataFrame([{
    "transport_gate_pass": transport_gate["transport_gate_pass"],
    "transport_min_train_coverage": transport_gate["transport_min_train_coverage"],
    "transport_min_test_coverage": transport_gate["transport_min_test_coverage"],
    "status": transport_gate["status"],
}]))


### Matriz de datasets Day 04.1

,variant_name,status,rows_output,events_output,added_columns_n,min_coverage_train,min_coverage_test,events_with_invalid_positive_count
0,competition,built,155946,15404,9,1.000000,1.000000,0
1,dispersion,built,155946,15404,5,1.000000,1.000000,0
2,source_quality,built,155946,15404,4,1.000000,1.000000,0
3,transport_only,built,155946,15404,9,0.866663,0.974966,0


### Detalle del gate de transport_only

,transport_gate_pass,transport_min_train_coverage,transport_min_test_coverage,status
0,True,0.866663,0.974966,built


## 4. Métricas puras por variante vs baseline

Aquí la comparación buena es 1:1 contra baseline oficial, con el mismo holdout temporal y la misma receta `LR_smote_0.5`. Si una familia no mejora el gate principal, no se recombina por intuición.


In [6]:
baseline = registry.loc[registry["model_role"] == "baseline"].tail(1).iloc[0]

metric_files = {
    "V2_SOURCE_QUALITY_LR_smote_0.5_v1": metrics_dir / "20260306T095152Z_V2_SOURCE_QUALITY_LR_smote_0.5_v1_metrics.json",
    "V2_DISPERSION_LR_smote_0.5_v1": metrics_dir / "20260306T095152Z_V2_DISPERSION_LR_smote_0.5_v1_metrics.json",
    "V2_COMPETITION_LR_smote_0.5_v1": metrics_dir / "20260306T095152Z_V2_COMPETITION_LR_smote_0.5_v1_metrics.json",
}

rows = []
for model_variant, path in metric_files.items():
    payload = json.loads(path.read_text(encoding="utf-8"))
    metrics = payload["metrics"]
    rows.append({
        "model_variant": model_variant,
        "top1_hit": metrics["top1_hit"],
        "top2_hit": metrics["top2_hit"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "f1_pos": metrics["f1_pos"],
        "coverage": metrics["coverage"],
        "delta_top2_vs_baseline": metrics["top2_hit"] - float(baseline["top2_hit"]),
        "delta_bal_acc_vs_baseline": metrics["balanced_accuracy"] - float(baseline["balanced_accuracy"]),
    })
metrics_df = pd.DataFrame(rows).sort_values(["top2_hit", "balanced_accuracy"], ascending=False).reset_index(drop=True)
display(metrics_df)


,model_variant,top1_hit,top2_hit,balanced_accuracy,f1_pos,coverage,delta_top2_vs_baseline,delta_bal_acc_vs_baseline
0,V2_DISPERSION_LR_smote_0.5_v1,0.764527,0.857197,0.864837,0.651170,1.0,-0.001139,-0.000247
1,V2_SOURCE_QUALITY_LR_smote_0.5_v1,0.762248,0.856817,0.854527,0.634240,1.0,-0.001519,-0.010557
2,V2_COMPETITION_LR_smote_0.5_v1,0.763768,0.854539,0.866691,0.667075,1.0,-0.003797,0.001607


## 5. Decisión sobre `V3_A2_SELECTED_SIGNALS`

La regla de recombinación es estricta: solo se construye una recombinación si alguna familia pura muestra mejora útil sin degradar el gate principal. Si ninguna lo logra, `V3_A2` no se crea.


In [7]:
display(pd.DataFrame([day041_summary]))
print("Conclusión:")
print("- `transport_only` queda excluido por cobertura histórica insuficiente en train.")
print("- Ninguna familia pura supera baseline en el gate principal.")
print("- No se construye `V3_A2_SELECTED_SIGNALS_LR_smote_0.5_v1`.")


,status,run_id,trained_variants,selected_families,selected_variant_trained,best_pure_variant,policy_variant,transport_variant_status,transport_gate_pass
0,ok,20260306T095152Z,"[V2_SOURCE_QUALITY_LR_smote_0.5_v1, V2_DISPERS...",[],,V2_DISPERSION_LR_smote_0.5_v1,V2_DISPERSION_LR_smote_0.5_WITH_DETERMINISTIC_...,excluded_parser_gate_failed,False


Conclusión:
- `transport_only` queda excluido por cobertura histórica insuficiente en train.
- Ninguna familia pura supera baseline en el gate principal.
- No se construye `V3_A2_SELECTED_SIGNALS_LR_smote_0.5_v1`.


## 6. Mejor variante pura + policy Day03

La mejor variante pura fue `V2_DISPERSION_LR_smote_0.5_v1`. Sobre esa única variante sí se ejecutó la policy de coherencia por albarán, porque el objetivo aquí era medir serving secundario sin mezclarlo con la decisión del reentreno.


In [8]:
policy_summary_path = metrics_dir / "20260306T095152Z_V2_DISPERSION_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json"
policy_summary = json.loads(policy_summary_path.read_text(encoding="utf-8"))
policy_metrics = policy_summary["metrics"]

policy_df = pd.DataFrame([{
    "model_variant": policy_summary["model_variant"],
    "top1_before": policy_metrics["top1_hit_before"],
    "top1_after": policy_metrics["top1_hit_after"],
    "top2_before": policy_metrics["top2_hit_before"],
    "top2_after": policy_metrics["top2_hit_after"],
    "coherence_before": policy_metrics["coherence_before"],
    "coherence_after": policy_metrics["coherence_after"],
    "overrides_count": policy_metrics["overrides_count"],
    "overrides_improved": policy_metrics["overrides_improved"],
    "overrides_harmed": policy_metrics["overrides_harmed"],
}])
display(policy_df)


,model_variant,top1_before,top1_after,top2_before,top2_after,coherence_before,coherence_after,overrides_count,overrides_improved,overrides_harmed
0,V2_DISPERSION_LR_smote_0.5_WITH_DETERMINISTIC_...,0.764527,0.786935,0.857197,0.857197,0.887671,1.0,59,59,0


## 7. Decisiones abiertas para Day 05

- Mantener como referencia operativa el baseline de Day 03 (`champion + capa determinista`).
- No abrir tuning todavía: la lectura causal de Day 04.1 es suficientemente clara sin mezclar receta y señal.
- Decidir si Brent directo (`V3_B`) entra ya en Day 05 o si se prioriza cierre de producto/demo.
- La línea de transporte queda técnicamente demostrada, pero no promocionable todavía por cobertura histórica insuficiente en train.
